In [ ]:
import os
import subprocess
from pathlib import Path

URL = (
    "https://www.jianyu360.cn/page_workDesktop/work-bench/page"
    "?link=https%3A%2F%2Fwww.jianyu360.cn"
    "%2Ffront%2FdataExport%2FtoSieve"
)


def find_chrome():
    # 查找 Windows 注册表中的 Chrome 安装路径
    import winreg

    registry_path = (
        r"SOFTWARE\Microsoft\Windows"
        r"\CurrentVersion\App Paths\chrome.exe"
    )

    for root in (winreg.HKEY_CURRENT_USER, winreg.HKEY_LOCAL_MACHINE):
        for view in (winreg.KEY_WOW64_64KEY, winreg.KEY_WOW64_32KEY):
            try:
                with winreg.OpenKey(
                    root, registry_path, 0, winreg.KEY_READ | view
                ) as key:
                    value, _ = winreg.QueryValueEx(key, "")
                    path = Path(os.path.expandvars(value.strip('"')))
                    if path.is_file():
                        return path
            except OSError:
                continue

    # 查找常见安装目录
    for variable in ("PROGRAMFILES", "PROGRAMFILES(X86)", "LOCALAPPDATA"):
        base = os.environ.get(variable)
        if base:
            path = Path(base) / "Google/Chrome/Application/chrome.exe"
            if path.is_file():
                return path

    raise FileNotFoundError("未找到谷歌浏览器，请确认已安装 Google Chrome。")


def main():
    chrome = find_chrome()
    subprocess.Popen([str(chrome), URL])
    print("已请求谷歌浏览器打开剑鱼标讯。")
    print("本程序现在退出，不会主动关闭浏览器。")


if __name__ == "__main__":
    try:
        main()
    except Exception as error:
        print(f"打开失败：{error}")
        input("按回车退出……")

In [ ]:
%pip install -U selenium

In [ ]:
import subprocess
import time
import socket
from pathlib import Path

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait

BASE = Path(r"D:\桌面\data")
PROFILE = BASE / "notebook_chrome"
DOWNLOADS = BASE / "downloads"

PROFILE.mkdir(parents=True, exist_ok=True)
DOWNLOADS.mkdir(parents=True, exist_ok=True)

HISTORY_URL = (
    "https://www.jianyu360.cn/page_workDesktop/"
    "work-bench/app/big/data_pack/history"
)

# 分配一个本机调试端口
with socket.socket() as sock:
    sock.bind(("127.0.0.1", 0))
    port = sock.getsockname()[1]

# Chrome 独立运行，不放在自动关闭浏览器的 with 语句中
chrome_process = subprocess.Popen([
    str(find_chrome()),
    f"--remote-debugging-port={port}",
    "--remote-debugging-address=127.0.0.1",
    f"--user-data-dir={PROFILE}",
    "--no-first-run",
    "--no-default-browser-check",
    HISTORY_URL,
])

# 等待 Chrome 启动
deadline = time.monotonic() + 30
while True:
    try:
        with socket.create_connection(("127.0.0.1", port), timeout=1):
            break
    except OSError:
        if time.monotonic() > deadline:
            raise RuntimeError("Chrome 调试端口未启动，请查看浏览器窗口。")
        time.sleep(0.5)

options = webdriver.ChromeOptions()
options.binary_location = str(find_chrome())
options.debugger_address = f"127.0.0.1:{port}"

# 首次连接可能需要联网下载匹配的 ChromeDriver
driver = webdriver.Chrome(options=options)

# 下载由 Chrome 直接保存，使用正常文件名
driver.execute_cdp_cmd("Browser.setDownloadBehavior", {
    "behavior": "allow",
    "downloadPath": str(DOWNLOADS.resolve()),
})

print("已连接 Chrome。")
print("请在新窗口登录剑鱼标讯，登录完成后运行下一个单元格。")

In [ ]:
"""在已建立 Selenium driver 连接的 Jupyter 单元格中运行。

用户于 2026-09-10 确认：Chrome 成功弹出测试提示。
本文件不是独立启动脚本，运行前必须存在 driver 变量。
"""

windows = driver.window_handles

if not windows:
    print("没有可用窗口，请把这个结果告诉我。")
else:
    driver.switch_to.window(windows[-1])
    driver.execute_script("alert('Python 已成功控制这个浏览器！');")
    print("请查看 Chrome，是否弹出了提示框。")
